# Task 11: Vision Transformer (ViT) Patch Projection and Multi-Head Self-Attention from Scratch

**Objective:** Adapt Transformer architectures to visual data by constructing spatial patch extraction modules, adding learnable class and position embeddings, and implementing multi-head self-attention using Einstein summation syntax via `einops`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import matplotlib.pyplot as plt
import numpy as np

# 1. Linear Patch Projection Layer
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, embed_dim=256):
        super().__init__()
        self.patch_size = patch_size
        # Projection of flattened patches to embedding dimension
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x):
        # Input: (B, C, H, W)
        x = self.projection(x) # Shape: (B, embed_dim, H/P, W/P)
        x = rearrange(x, 'b e h w -> b (h w) e') # Shape: (B, N_patches, embed_dim)
        return x

# 2. Multi-Head Self-Attention (MHSA) using einops and einsum
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        # QKV Projections
        self.qkv_projection = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_projection = nn.Linear(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        
        # Save attention weights for visualization
        self.attention_weights = None
        
    def forward(self, x):
        B, N, E = x.shape
        
        # Calculate QKV matrices
        qkv = self.qkv_projection(x) # (B, N, 3*E)
        # Split queries, keys, and values and project into heads using einops
        q, k, v = rearrange(qkv, 'b n (qkv h d) -> qkv b h n d', qkv=3, h=self.num_heads, d=self.head_dim)
        
        # Compute scaled attention weights using Einstein Summation
        # Attention = Softmax(Q K^T / sqrt(d_k))
        scores = torch.einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        # Cache for visualization
        self.attention_weights = attn.detach()
        
        # Compute weighted sum of values
        out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        # Concatenate heads and project
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.out_projection(out)
        return out

In [ ]:
# 3. Full Transformer Block and test pass with attention extraction
class ViTBlock(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        
    def forward(self, x):
        # Residual connection + pre-LN self-attention
        x = x + self.attn(self.ln1(x))
        # Residual connection + pre-LN feed forward network
        x = x + self.mlp(self.ln2(x))
        return x

# Verify layout using dummy image
img = torch.randn(2, 3, 224, 224) # batch size 2, 224x224 RGB image
patch_size = 16
embed_dim = 256

pat_embed = PatchEmbedding(in_channels=3, patch_size=patch_size, embed_dim=embed_dim)
vit_block = ViTBlock(embed_dim=embed_dim, num_heads=8)

patches = pat_embed(img)

# Append Class Token
cls_token = nn.Parameter(torch.randn(2, 1, embed_dim))
patches = torch.cat([cls_token, patches], dim=1)

# Add Position Embeddings
pos_embed = nn.Parameter(torch.randn(1, 1 + (224//patch_size)**2, embed_dim))
x = patches + pos_embed

# Feed to ViT block
out = vit_block(x)

print("Patches shape:         ", patches.shape)
print("Class-token & pos shape:", x.shape)
print("Output shape:          ", out.shape)
print("Self-Attention weight shape:", vit_block.attn.attention_weights.shape)

assert out.shape == x.shape, "Verification failed! Shape mismatch!"
print("Success: Vision Transformer embedding projection and self-attention operations completed successfully!")